In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
dbutils.widgets.text('externalfiles_zone', 'None', '07-Externalfiles Zone')
externalfiles_zone = dbutils.widgets.get('externalfiles_zone')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_ext_lsbr23t_extfile(
    spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #        "ExtLSBR23TId string,"
        #        "SourceSystem  string,"
        #        "PortfolioKey  string,"
        #        "PortfolioName string,"
        #        "IngestionTimestamp    timestamp,"
        #        "BatchId   string,"
        #        "ExtLSBR23TKey    string,"
        #        "EntityVersion string,"
        #        "ExposureKey   string,"
        #        "AgreementKey  string,"
        #        "CollateralCategory    string,"
        #        "ExposureSourceSystem  string,"
        #        "BusinessDate  date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.ExtLSBR23T"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "extfile",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== LSBR23T ==================
        LSBR23T_extra_filters = None

        if LSBR23T_extra_filters is not None:
            LSBR23T_extra_filters.append(date_filter)
            LSBR23T_final_filters = " AND ".join(LSBR23T_extra_filters)
        else:
            LSBR23T_final_filters = date_filter

            LSBR23T_selected_fields = [
                "ExposureKey",
                "AgreementKey",
                "CollateralCategory",
                "SourceSystem",
                "businessdate",
            ]

            LSBR23T = load_landed_delta_table(
                spark,
                externalfiles_zone,  # type: ignore
                "extlsbr23t",
                LSBR23T_final_filters,
                LSBR23T_selected_fields,
            )

        # ================== LSBR23T_extfile_df ==================
        LSBR23T_extfile_primary_key_fields = [
            LSBR23T.ExposureKey,
            LSBR23T.AgreementKey,
            LSBR23T.CollateralCategory,
            LSBR23T.SourceSystem,
            lit("extfile"),
        ]
        LSBR23Tkey_extfile_primary_key_fields = [
            LSBR23T.ExposureKey,
            LSBR23T.AgreementKey,
            LSBR23T.CollateralCategory,
            LSBR23T.SourceSystem,
            lit("extfile"),
        ]

        LSBR23T_extfile_df = (
            LSBR23T.withColumn(
                "ExtLSBR23TID",
                sha2(concat_ws(";", *LSBR23T_extfile_primary_key_fields), 256),
            )
            .withColumn(
                "PortfolioKey",
                (lit("-9999")),
            )
            .withColumn(
                "PortfolioName",
                (lit("Core Retail")),
            )
            .withColumn(
                "ExtLSBR23TKey", concat_ws(";", *LSBR23Tkey_extfile_primary_key_fields)
            )
            .withColumn("LSBR23TStatus", lit(None).cast("string"))
            .withColumn("DebtorPartyKey", lit(None).cast("string"))
        )

        LSBR23T_extfile_df = LSBR23T_extfile_df.select(
            LSBR23T_extfile_df.ExtLSBR23TID.cast("string"),
            lit("extfile").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            LSBR23T_extfile_df.PortfolioKey,
            LSBR23T_extfile_df.PortfolioName,
            LSBR23T_extfile_df.ExtLSBR23TKey.cast("string").alias("ExtLSBR23TKey"),
            LSBR23T_extfile_df.ExposureKey.cast("string").alias("ExposureKey"),
            LSBR23T_extfile_df.AgreementKey.cast("string").alias("AgreementKey"),
            LSBR23T_extfile_df.CollateralCategory.cast("string").alias(
                "CollateralCategory"
            ),
            LSBR23T_extfile_df.SourceSystem.alias("ExposureSourceSystem"),
            LSBR23T_extfile_df.businessdate.alias("BusinessDate"),
        )

        # ================== Append ==================

        LSBR23T_extfile_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                externalfiles_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_ext_lsbr23t_extfile(spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix)